In [43]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter.BayesianFunctionalRegression import BayesianFunctionalRegression
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.t import TDistribution
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample

from pyro.infer import Predictive
import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pyro.set_rng_seed(1)

%matplotlib inline
plt.style.use('default')


In [12]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)

X = df[["cont_africa","rugged","cont_africa_x_rugged"]]
y = df[["rgdppc_2000"]]

x_data, y_data = data[:, :-1], data[:, -1]


In [49]:
class UserCustomModel(PyroModule):
    def __init__(self, in_features, mu=0.0, sigma=1.0):
        super().__init__()

        self.linear = PyroModule[nn.Linear](in_features, 1)

        self.linear.weight = PyroSample(
            dist.Normal(mu, sigma)
            .expand([1, in_features])
            .to_event(2)
        )

        self.linear.bias = PyroSample(
            dist.Normal(mu, sigma)
            .expand([1])
            .to_event(1)
        )

    def forward(self, x, y=None):
        obs_scale = pyro.sample(
            "obs_scale",
            dist.HalfNormal(2.0)
        )

        # Student-t
        df = 2.0 + pyro.sample(
            "df_minus_two",
            dist.Exponential(1.0)
        )

        mean = self.linear(x).squeeze(-1)

        with pyro.plate("data", x.shape[0]):
            pyro.sample(
                "obs",
                dist.StudentT(
                    df=df,
                    loc=mean,
                    scale=obs_scale,
                ),
                obs=y,
            )

        return mean


In [50]:
class UserCustomGuide(PyroModule):
    def __init__(
        self,
        model,
        in_features,
        posterior_df=4.0,
        init_scale=0.1,
    ):
        super().__init__()

        self.model = model
        self.in_features = in_features

        self.register_buffer(
            "posterior_df",
            torch.tensor(float(posterior_df)),
        )

        self.weight_loc = nn.Parameter(
            torch.zeros(1, in_features)
        )
        self.weight_scale = PyroParam(
            torch.full((1, in_features), init_scale),
            constraint=constraints.positive,
        )

        self.bias_loc = nn.Parameter(
            torch.zeros(1)
        )
        self.bias_scale = PyroParam(
            torch.full((1,), init_scale),
            constraint=constraints.positive,
        )

        self.obs_scale_loc = nn.Parameter(
            torch.tensor(0.0)
        )
        self.obs_scale_scale = PyroParam(
            torch.tensor(init_scale),
            constraint=constraints.positive,
        )

        self.df_loc = nn.Parameter(
            torch.tensor(1.0)
        )
        self.df_scale = PyroParam(
            torch.tensor(init_scale),
            constraint=constraints.positive,
        )

    def forward(self, x, y=None):
        weight = pyro.sample(
            "linear.weight",
            dist.StudentT(
                df=self.posterior_df,
                loc=self.weight_loc,
                scale=self.weight_scale,
            ).to_event(2),
        )

        bias = pyro.sample(
            "linear.bias",
            dist.StudentT(
                df=self.posterior_df,
                loc=self.bias_loc,
                scale=self.bias_scale,
            ).to_event(1),
        )

        obs_scale = pyro.sample(
            "obs_scale",
            dist.TransformedDistribution(
                dist.StudentT(
                    df=self.posterior_df,
                    loc=self.obs_scale_loc,
                    scale=self.obs_scale_scale,
                ),
                [transforms.SoftplusTransform()],
            ),
        )

        df_minus_two = pyro.sample(
            "df_minus_two",
            dist.TransformedDistribution(
                dist.StudentT(
                    df=self.posterior_df,
                    loc=self.df_loc,
                    scale=self.df_scale,
                ),
                [transforms.SoftplusTransform()],
            ),
        )
        self.info_ = {
            "linear.weight": weight,
            "linear.bias": bias,
            "obs_scale": obs_scale,
            "df_minus_two": df_minus_two,
        }

        return {
            "linear.weight": weight,
            "linear.bias": bias,
            "obs_scale": obs_scale,
            "df_minus_two": df_minus_two,
        }


In [ ]:
def pyro2skpro(df=4.0):
    """
    Contract parameter.predict_proba() -> Skpro's distributions
    So, converter class from pyro.dist -> to skpro.dist
    """
    def converter(y_samples, num_samples, X_array, index, columns):
        n_instances = X_array.shape[0]

        y_samples = y_samples.reshape(num_samples, n_instances)

        pred_mean = y_samples.mean(dim=0)
        pred_std = y_samples.std(dim=0, unbiased=False)

        pred_sigma = pred_std * ((df - 2.0) / df) ** 0.5

        pred_mean = pred_mean.detach().cpu().numpy()
        pred_sigma = pred_sigma.detach().cpu().numpy()

        return TDistribution(
            mu=pred_mean.reshape(-1, 1),
            sigma=pred_sigma.reshape(-1, 1),
            df=df,
            index=index,
            columns=columns,
        )

    return converter


In [52]:
model = UserCustomModel(in_features=3)
guide = UserCustomGuide(model, 3)


In [53]:
linear_model = BayesianFunctionalRegression(
    model=model,
    guide=guide,
    converter=pyro2skpro(),
    num_iterations=500, 
    lr=0.03,
    posterior_samples=100,
) 


In [54]:
linear_model.fit(X, y)


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\pyro\util.py:288: UserWarning: Found non-auxiliary vars in guide but not model, consider marking these infer={'is_auxiliary': True}:
{'linear.weight', 'linear.bias'}
  warnings.warn(
c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\pyro\util.py:303: UserWarning: Found vars in model but not guide: {'model.linear.weight', 'model.linear.bias'}
  warnings.warn(f"Found vars in model but not guide: {bad_sites}")


[iteration 0001] loss: 9.6276
[iteration 0101] loss: 4.1806
[iteration 0201] loss: 4.4751
[iteration 0301] loss: 3.8867
[iteration 0401] loss: 3.7749


BayesianFunctionalRegression(converter=<function pyro2skpro.<locals>.converter at 0x0000017B6840B7E0>,
                             guide=UserCustomGuide(
  (model): UserCustomModel(
    (linear): PyroLinear(in_features=3, out_features=1, bias=True)
  )
),
                             model=UserCustomModel(
  (linear): PyroLinear(in_features=3, out_features=1, bias=True)
),
                             num_iterations=500, posterior_samples=100)

In [55]:
linear_model.guide.info_

# {'linear.weight': tensor([[ 583846.8750,  -70176.7891, -173804.8281]], grad_fn=<AddBackward0>),
#  'linear.bias': tensor([-554430.6875], grad_fn=<AddBackward0>),
#  'obs_scale': tensor(5.7112, grad_fn=<SoftplusBackward0>),
#  'df_minus_two': tensor(0.2364, grad_fn=<SoftplusBackward0>)}

# regression function:
# y ∼ StudentT(
#     df=2.2364,
#     loc=-554430.6875
#         + 583846.8750*x1
#         - 70176.7891*x2
#         - 173804.8281*x1*x2,
#     scale=5.7112
# )


{'linear.weight': tensor([[ 542001.7500, -369536.3750,  205117.1719]], grad_fn=<AddBackward0>),
 'linear.bias': tensor([-293316.5625], grad_fn=<AddBackward0>),
 'obs_scale': tensor(5.8765, grad_fn=<SoftplusBackward0>),
 'df_minus_two': tensor(0.4411, grad_fn=<SoftplusBackward0>)}

In [56]:
import pandas as pd

X_df = pd.DataFrame(
    x_data[:5].detach().cpu().numpy(),
    columns=["x1", "x2", "x3"],
)

pred_dist = linear_model.predict_proba(X_df)


In [57]:
pred_dist


TDistribution(columns=Index(['y'], dtype='object'), df=4.0,
              index=RangeIndex(start=0, stop=5, step=1),
              mu=array([[ 0.8270459 ],
       [-0.30934802],
       [ 0.7100767 ],
       [-0.637819  ],
       [-1.67604   ]], dtype=float32),
              sigma=array([[6.860208 ],
       [6.338256 ],
       [7.8022914],
       [9.935845 ],
       [9.930204 ]], dtype=float32))

In [58]:
pred_dist.mu.shape


(5, 1)